# ScoutOSS cohort analysis

All data here is **de-identified**: patients are HMAC pseudonyms, dates are shifted per project, ages over 89 are reported as 90. Never paste identifiers into notebooks.

1. Query the read-only OpenSearch indices (`dicom_studies`, `dicom_patients`, `clinical_events`).
2. Pull imaging from XNAT with **pyxnat** for the same sessions.

In [ ]:
import os

import pandas as pd
from opensearchpy import OpenSearch

os_client = OpenSearch(hosts=[os.environ["SCOUTOSS_OPENSEARCH"]])
resp = os_client.search(
    index="dicom_studies",
    body={
        "size": 1000,
        "query": {
            "bool": {"filter": [{"term": {"modalities": "CT"}}, {"term": {"body_parts": "CHEST"}}]}
        },
    },
)
studies = pd.DataFrame([h["_source"] for h in resp["hits"]["hits"]])
studies[
    ["patient_pseudonym", "study_date", "patient_age", "patient_sex", "xnat_experiment_id"]
].head()

In [ ]:
studies["patient_age"].plot.hist(bins=range(0, 100, 10), title="Age at CT chest (years)");

## Download images with pyxnat

Use your own XNAT account (project membership controls what you can read).

In [ ]:
from getpass import getpass

from pyxnat import Interface

xnat = Interface(server=os.environ["SCOUTOSS_XNAT"], user=input("XNAT user: "), password=getpass())
exp_id = studies["xnat_experiment_id"].dropna().iloc[0]
experiment = xnat.select.experiment(exp_id)
print(experiment.label(), [s.id() for s in experiment.scans()])
# experiment.scans().download('/home/jovyan/data', type='ALL', extract=True)